In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [14]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print(f"Path to dataset files: {path}/spam.csv")

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Path to dataset files: /kaggle/input/sms-spam-collection-dataset/spam.csv


In [15]:
df = pd.read_csv("/kaggle/input/sms-spam-collection-dataset/spam.csv", encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [16]:
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.rename(columns={'v1': 'label', 'v2': 'message'}, inplace=True)
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [18]:
X = df['message'].fillna('')
y = df['label']

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [20]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

In [21]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [22]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

BNB = BernoulliNB()
BNB.fit(X_train_cv, y_train)
yprid = BNB.predict(X_test_cv)

accuracy = accuracy_score(y_test, yprid)
print("Accuracy:", accuracy)

Cr = classification_report(y_test, yprid)
print("Classification Report:\n", Cr)

Cm = confusion_matrix(y_test, yprid)
print("Confusion Matrix:\n", Cm)

train = BNB.score(X_train_cv, y_train)
print("Training Score:", train)

test = BNB.score(X_test_cv, y_test)
print("Testing Score:", test)

Accuracy: 0.9757847533632287
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.82      0.90       149

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115

Confusion Matrix:
 [[966   0]
 [ 27 122]]
Training Score: 0.9878842270585596
Testing Score: 0.9757847533632287


In [24]:
import pandas as pd

# Take input for the message
user_input = input("Enter message: ")

# Transform the input using the existing CountVectorizer
input_vectorized = cv.transform([user_input])

# Make prediction using the trained MultinomialNB model (MNB)
pred_numeric = BNB.predict(input_vectorized)[0]
probability = BNB.predict_proba(input_vectorized)[0]

# Decode the predicted numeric class back to its original label
prediction = le.inverse_transform([pred_numeric])[0]
class_labels = le.inverse_transform(BNB.classes_)

print(f"\n\t Prediction: {prediction}")

# Print the probabilities for each class
print("\t Class Probabilities:")
for class_name, prob in zip(class_labels, probability):
    print(f"\t - {class_name}: {round(prob * 100, 2)}%")

Enter message: You got 

	 Prediction: ham
	 Class Probabilities:
	 - ham: 100.0%
	 - spam: 0.0%
